# Vector + Graph RAG — VectorCypherRetriever

벡터 검색과 그래프 탐색을 **결합** 한다. **`VectorCypherRetriever`** 는:
1. 먼저 **벡터 유사도** 로 관련 노드를 찾고,
2. 그 노드에서 **Cypher 로 관계를 확장** 해 추가 정보를 가져온다.

예: "마법 보드게임 영화"(줄거리 벡터 검색) → 그 영화의 **배우/장르**(관계 탐색)까지 한 번에.

> Neo4j + `OPENAI_API_KEY` 필요. 예시는 영화 그래프 (plot 임베딩 + Actor-ACTED_IN->Movie).

## Neo4j 연결 & LLM
`.env` 에 `NEO4J_URI` / `NEO4J_USERNAME` / `NEO4J_PASSWORD` / `OPENAI_API_KEY` 를 넣는다. (README 의 'Neo4j 준비' 참고)

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
URI = os.environ["NEO4J_URI"]
AUTH = (os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Neo4j 연결 성공")

## 1. 임베딩 + 벡터 인덱스
로컬 임베딩 모델(SentenceTransformers)도 쓸 수 있다 (OpenAI 임베딩 대신, 비용/오프라인 이점).

In [ ]:
from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings

# 로컬 임베딩 모델 (첫 실행 시 다운로드)
embedder = SentenceTransformerEmbeddings(model="all-MiniLM-L6-v2")

In [ ]:
from neo4j_graphrag.indexes import create_vector_index

INDEX_NAME = "plotindex"
# plot 임베딩이 Movie 노드에 있다고 가정 (차원은 임베딩 모델에 맞춤: MiniLM=384)
create_vector_index(
    driver, INDEX_NAME, label="Movie",
    embedding_property="plotEmbedding", dimensions=384, similarity_fn="cosine",
)

## 2. VectorCypherRetriever — 벡터 검색 + 관계 확장

`retrieval_query` 로 **벡터로 찾은 노드(`node`)에서 이어갈 Cypher** 를 지정한다. 여기서는 찾은 영화의 배우들까지 함께 가져온다.

In [ ]:
from neo4j_graphrag.retrievers import VectorCypherRetriever

# node = 벡터 검색으로 찾은 Movie 노드. 거기서 배우 관계를 확장
retrieval_query = """
MATCH (actor:Actor)-[:ACTED_IN]->(node)
RETURN node.title AS movie_title,
       node.plot AS movie_plot,
       collect(actor.name) AS actors
"""

retriever = VectorCypherRetriever(
    driver, index_name=INDEX_NAME,
    retrieval_query=retrieval_query, embedder=embedder,
)

result = retriever.search(query_text="a movie about a magic board game", top_k=1)
print(result)

## 3. GraphRAG + 커스텀 프롬프트
`RagTemplate` 으로 답변 형식을 지정할 수 있다 (제목/줄거리/장르/배우 포함, 한국어 등).

In [ ]:
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import RagTemplate, GraphRAG

prompt_template = RagTemplate(
    template=(
        "You are a helpful movie assistant. Based on the question and retrieved movie info, "
        "identify the most relevant movie and explain it clearly in Korean.\n"
        "Include: title, brief plot, genre(s), main actor(s).\n\n"
        "Question: {query_text}\n\nContext: {context}\n\nAnswer:"
    ),
    expected_inputs=["context", "query_text"],
)

llm = OpenAILLM(model_name="gpt-4o")
graph_rag = GraphRAG(retriever, llm, prompt_template=prompt_template)

response = graph_rag.search(query_text="마법 보드게임에 대한 영화가 뭐야?", return_context=True)
print(response.answer)

## 정리

- **VectorCypherRetriever** = 벡터 검색(의미) + Cypher 확장(관계) 결합
- `retrieval_query` 로 찾은 노드에서 관계를 얼마나 확장할지 지정
- `RagTemplate` 으로 답변 형식·언어 커스터마이징
- 세 방식 비교:
  - **Vector**(01): 의미 유사 텍스트
  - **Graph/Text2Cypher**(02): 관계형 질의
  - **Vector+Graph**(03): 의미로 찾고 관계로 확장

다음: 이 Text2Cypher 를 LangGraph 로 직접 구현하고 자가교정까지 붙인 **GraphRAG Agent**(04).